# 00 · First look at the USAspending sample

**Goal (PLAN.md milestone 1):** load a small federal contract sample and explain every column.

Before running this, collect and flatten a snapshot from the repo root:

```bash
python -m govspend.collect --max-records 200
python -m govspend.flatten
```

This is a **development sample**, sorted by award amount — it over-represents large awards and is not the study cohort.

In [ ]:
import json, sys
from pathlib import Path

import pandas as pd

REPO = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(REPO / 'src'))

from govspend.flatten import latest_run, load_records, flatten

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

## Collection notes
What was requested, when, and why collection stopped.

In [ ]:
run_dir = latest_run(REPO / 'data' / 'raw')
notes = json.loads((run_dir / 'collection_notes.json').read_text())
{k: notes[k] for k in ['run_id', 'retrieved_at_utc_start', 'record_cap', 'records_collected', 'stop_reason', 'sort', 'order', 'time_period_semantics']}

## Load and flatten

In [ ]:
raw = load_records(run_dir)
df = flatten(raw)
print(f'{len(raw)} raw results -> {len(df)} awards ({df.attrs["duplicates_dropped"]} duplicates dropped)')
df.dtypes

## Five records
Check a few of these against usaspending.gov — the award page URL is `https://www.usaspending.gov/award/<generated_internal_id>`.

In [ ]:
sample = df.sample(5, random_state=513) if len(df) >= 5 else df
sample.assign(source_url='https://www.usaspending.gov/award/' + sample['generated_internal_id'])

## Missing values per column

In [ ]:
df.isna().sum().to_frame('n_missing').assign(pct=lambda t: (100 * t.n_missing / len(df)).round(1))

## Award amount
Counts of zero/negative amounts matter before any log transform (PLAN.md §4).

In [ ]:
amt = df['award_amount']
print(amt.describe().apply(lambda v: f'{v:,.0f}'))
print('\nzero:', (amt == 0).sum(), '| negative:', (amt < 0).sum(), '| missing:', amt.isna().sum())

## Categorical coverage

In [ ]:
for col in ['awarding_agency', 'naics_code', 'contract_award_type']:
    print(f'\n--- {col} ---')
    print(df[col].value_counts(dropna=False).head(10))

## Observations

_Write 3–5 sentences here once real data is loaded: what the sample covers, anything surprising, and which columns need a precise definition in `docs/data_dictionary.md`._